# Cumulative Spend Distribution Analysis

This notebook characterizes the underlying cumulative spend-position distribution that feeds the later Beta CDF modeling work. It should be read before `clustered_curve_beta_cdf_model.ipynb`: this page explains the empirical shape of `CUMULATIVEBURNPCT` as a function of `ELAPSEDPCT`, while the later notebook evaluates predictive reference curves against held-out items.

## Dataset and Overall Character

| Metric | Value |
| --- | --- |
| Input CSV | clustered_data_input_allcustomers.csv |
| Cluster curve rows | 9,303 |
| Items | 1,003 |
| Train rows used for distribution fitting | 7,542 |
| Train items | 814 |
| Mean cumulative spend pct | 61.43% |
| Median cumulative spend pct | 65.57% |
| Std dev cumulative spend pct | 32.15% |
| Share at completed edge near 100% | 12.73% |
| Share at zero edge near 0% | 0.21% |
| Pearson corr elapsed vs cumulative spend | 0.7664 |
| Spearman corr elapsed vs cumulative spend | 0.7878 |

In [ ]:
import csv
from pathlib import Path

path = Path('clustered_data_input.csv')
with path.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
print(path)
print(len(rows))
print(reader.fieldnames)

## Marginal Distribution of Cumulative Spend

The marginal distribution is bounded between 0 and 1 and is strongly affected by repeated observations from the same item curve. It is not a standalone time-free distribution: a point at 90% elapsed and a point at 20% elapsed should not be expected to have the same cumulative spend behavior. The visible pile-up near 100% is expected because every completed item contributes a final cluster at full cumulative spend.

| Quantile | Cumulative spend pct |
| --- | --- |
| p01 | 0.75% |
| p05 | 5.95% |
| p10 | 13.01% |
| p25 | 34.03% |
| p50 | 65.57% |
| p75 | 93.33% |
| p90 | 100.00% |
| p95 | 100.00% |
| p99 | 100.00% |



## Joint Distribution With Elapsed Percent

The joint view is the core characterization. The distribution is bounded, monotonic in expectation, heteroscedastic, and edge-inflated near completion. The diagonal line is the pure linear spend reference; density above the line is front-loaded spend, while density below it is back-loaded or delayed spend.



## Conditional Distribution by Elapsed Bucket

The table and band plot summarize `CUMULATIVEBURNPCT | ELAPSEDPCT bucket`. The widening and narrowing of the percentile bands matters more than the overall histogram because the production question is where an item sits relative to expected cumulative spend at its current elapsed position.

| Elapsed bucket | Rows | Mean | Median | P10 | P25 | P75 | P90 | IQR | Skew |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0.0-0.1 | 644 | 16.56% | 10.96% | 1.32% | 4.23% | 23.82% | 35.87% | 19.58% | 2.251 |
| 0.1-0.2 | 875 | 32.59% | 26.42% | 6.14% | 14.29% | 45.51% | 67.51% | 31.22% | 0.957 |
| 0.2-0.3 | 845 | 43.79% | 40.00% | 12.01% | 24.87% | 60.34% | 84.64% | 35.47% | 0.475 |
| 0.3-0.4 | 775 | 51.29% | 50.00% | 21.67% | 34.29% | 67.22% | 88.38% | 32.93% | 0.147 |
| 0.4-0.5 | 710 | 59.73% | 60.04% | 27.47% | 42.86% | 78.54% | 95.00% | 35.68% | -0.167 |
| 0.5-0.6 | 690 | 64.85% | 66.66% | 31.88% | 50.15% | 84.09% | 97.11% | 33.94% | -0.471 |
| 0.6-0.7 | 638 | 72.96% | 75.47% | 44.39% | 60.89% | 89.20% | 98.22% | 28.31% | -0.798 |
| 0.7-0.8 | 573 | 79.24% | 83.00% | 55.92% | 70.45% | 94.00% | 98.25% | 23.55% | -1.320 |
| 0.8-0.9 | 559 | 86.67% | 90.29% | 67.60% | 81.17% | 96.98% | 99.44% | 15.81% | -1.928 |
| 0.9-1.0 | 1,233 | 97.69% | 100.00% | 92.19% | 99.28% | 100.00% | 100.00% | 0.72% | -5.370 |

## Percentile Bands and Reference Curves

The empirical median curve is generally above the linear reference for much of the item life, which means these clustered payment curves are often front-loaded relative to time. The Beta CDF and anchored polynomial are compact parameterizations of that empirical shape; the later model notebook evaluates the Beta CDF approach more formally.



## Bucketed Density View

This view shows the full shape within each elapsed bucket rather than only percentiles. It makes the conditional spread visible: some elapsed regions are broad and multi-modal because items can receive lumpy clustered payments at different points in their lifecycle.



## Curve Parameterizations

| Model | Alpha / coefficients | Beta | MAE | RMSE | Bias | Clip share | Monotonic violations |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Beta CDF | 0.533314 | 0.806125 | 0.1495 | 0.2045 | 0.0014 |  |  |
| Anchored polynomial degree 3 | 1.008832, 1.417902, -1.748682 |  | 0.1487 | 0.2048 | -0.0039 | 0.0080 | 0 |
| Anchored polynomial degree 4 | 0.994063, 1.851193, -3.814823, 2.230702 |  | 0.1493 | 0.2042 | -0.0006 | 0.0000 | 0 |

The Beta CDF is a useful production candidate because it is naturally bounded and monotone. The anchored polynomial is useful as a flexible descriptive reference, but it is less constrained structurally and should be treated as diagnostic unless monotonicity and stability are explicitly checked.

## Residual Distribution Around Reference Curves

Residuals are `actual cumulative pct - expected cumulative pct`. A good reference curve centers this distribution closer to zero and reduces asymmetric bias. The residual plot shows why the cumulative spend model should not be purely linear: the linear reference leaves a larger systematic position offset where the empirical spend curve is front-loaded.



## Stratification by Duration

Duration buckets have visibly different median cumulative spend curves. This supports the later duration-bucket Beta CDF model: the expected curve is not fully universal across short and long items.



## Stratification by Cluster Count

Cluster count is another proxy for payment cadence and lumpiness. Items with fewer clusters tend to show coarser jumps, while higher-cluster items provide smoother cumulative curves.



## Interpretation

The cumulative spend distribution is best understood as a conditional bounded distribution, not as a single ordinary marginal distribution. Its important properties are:

- Bounded support at `[0, 1]`, with a structural edge at 100% from final clusters.
- Strong positive relationship between elapsed percent and cumulative spend percent.
- A median curve that is not purely linear, indicating systematic front-loading in the clustered payment data.
- Wide conditional dispersion, especially in the middle of the lifecycle, caused by lumpy payment postings and differing payment cadence.
- Meaningful stratification by item duration and cluster count.

This motivates the next-stage notebook: fit expected-position curves and evaluate whether Beta CDF parameterizations outperform a transparent linear reference.